# Data Science Workflow — Geldwäschereiüberwachung mit Dummy-Daten

**CAS AI Management & Strategy for Financial Services — Modul 2**

Dieses Notebook führt einen kompletten Data-Science-Workflow entlang von **CRISP-DM** durch.
Use Case: **Transaktionsüberwachung** in der Geldwäschereibekämpfung (AML). Wir erzeugen
synthetische Transaktionsdaten und vergleichen drei Wege, Verdachtsfälle (Alerts) zu erzeugen:

1. **Regelbasiert** — klassische Schwellenwert- und Musterregeln
2. **Statistisch** — Kennzahlen wie der z-Score (Abweichung vom eigenen Normalverhalten)
3. **Unsupervised (ML)** — Anomalieerkennung mit dem Isolation Forest

Am Schluss bewerten wir die Ansätze mit **Confusion Matrix, Precision, Recall und F1**.

> **Ziel der Session:** Sie müssen kein:e Data Scientist:in werden. Sie sollen aber den
> Werkzeugkasten und die Denkweise verstehen — damit Sie als Projektleiter:in mitreden,
> Ergebnisse einordnen und die richtigen Fragen stellen können.

> **Bedienung:** Markieren Sie eine Zelle und drücken Sie `Shift`+`Enter`, um sie auszuführen.
> Führen Sie die Zellen der Reihe nach von oben nach unten aus. Mit `Aufgabe`
> markierte Stellen sind zum Selber-Ausprobieren gedacht.

## CRISP-DM — der rote Faden

Der **Cross-Industry Standard Process for Data Mining** strukturiert fast jedes
Data-Science-Projekt in sechs Phasen. Wir gehen sie heute der Reihe nach durch:

| Phase | Was passiert | Wo im Notebook |
|---|---|---|
| 1. Business Understanding | Problem, Ziel, regulatorischer Rahmen | Abschnitt 1 |
| 2. Data Understanding | Daten sichten, EDA | Abschnitt 3 |
| 3. Data Preparation | Features bauen (Feature Engineering) | Abschnitt 4 |
| 4. Modeling | Modelle / Regeln anwenden | Abschnitt 5–7 |
| 5. Evaluation | Güte messen, mit Fachseite spiegeln | Abschnitt 8–9 |
| 6. Deployment | In Betrieb nehmen, überwachen | Diskussion / Folien |

In der Praxis ist das **kein Wasserfall**, sondern ein Kreislauf: Erkenntnisse aus der
Evaluation werfen einen oft zurück zur Datenaufbereitung.

## 0 — Setup

Wir laden die benötigten Bibliotheken. `pandas` für Tabellen, `numpy` für Zahlen,
`matplotlib` für Grafiken, `scikit-learn` für das ML-Modell und die Kennzahlen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
RNG = np.random.default_rng(42)   # fixer Zufallsstartwert => reproduzierbar
print("Setup ok")

## 1 — Business Understanding

**Worum geht es?** Banken sind gesetzlich verpflichtet, Transaktionen auf Hinweise für
Geldwäscherei zu überwachen (in der Schweiz: GwG, GwV-FINMA, VSB). Auffällige Vorgänge
münden in einen **Alert**, den ein:e Analyst:in (Compliance) prüft. Bestätigt sich der
Verdacht, folgt ggf. eine Verdachtsmeldung an die Meldestelle (MROS).

**Typische verdächtige Muster:**
- **Structuring / Smurfing** — grosse Beträge in viele kleine Tranchen knapp unter der
  Meldeschwelle stückeln.
- **Layering** — Geld schnell durch Konten und über Grenzen schleusen, oft in
  Hochrisikoländer, um die Herkunft zu verschleiern.
- **Plötzliche Spikes** — Aktivität, die stark vom bisherigen Verhalten der Kundin abweicht.

**Die zentrale Spannung:** Wir wollen möglichst **alle** echten Fälle finden (hoher *Recall*),
ohne die Analyst:innen mit **Fehlalarmen** zu überfluten (hohe *Precision*). Jeder False
Positive kostet Ermittlungszeit. Genau dieses Spannungsfeld machen wir heute messbar.

## 2 — Daten generieren (Dummy-Daten)

Echte Transaktionsdaten dürfen das Haus nicht verlassen — deshalb erzeugen wir synthetische
Daten, die den Charakter echter Daten nachbilden. Wir simulieren 800 Kund:innen mit
normalem Zahlungsverhalten und schleusen bei rund 4 % bekannte verdächtige Muster ein.
Diese **Ground Truth** (`label`) kennen wir nur, weil wir die Daten selbst gebaut haben —
in der Realität ist sie das Teuerste und Seltenste überhaupt.

In [ ]:
N_KUNDEN      = 800
START         = pd.Timestamp("2026-01-01")
TAGE          = 180
MELDESCHWELLE = 10_000     # CHF (didaktisch vereinfacht)

LAENDER_NORMAL = ["CH", "DE", "AT", "FR", "IT", "ES", "NL", "US"]
LAENDER_RISIKO = ["RU", "IR", "KP", "PA", "KY", "VG"]   # Hochrisiko (nur Beispiel)
KANAELE        = ["E-Banking", "Karte", "Schalter", "Dauerauftrag"]

def normale_transaktionen(kid):
    # Erzeugt 'unauffaellige' Transaktionen fuer eine:n Kund:in
    mu, sigma = RNG.uniform(6.0, 8.0), RNG.uniform(0.4, 0.9)   # individuelles Niveau
    n = RNG.integers(20, 60)
    return pd.DataFrame({
        "kunde_id":  kid,
        "tag":       RNG.integers(0, TAGE, n),
        "betrag":    np.round(RNG.lognormal(mu, sigma, n), 2),
        "land":      RNG.choice(LAENDER_NORMAL, n,
                                p=[0.55,0.12,0.06,0.08,0.07,0.04,0.04,0.04]),
        "richtung":  RNG.choice(["ein", "aus"], n),
        "kanal":     RNG.choice(KANAELE, n, p=[0.5,0.3,0.1,0.1]),
    })

zeilen, label = [], {}
for kid in range(N_KUNDEN):
    zeilen.append(normale_transaktionen(kid))
    label[kid] = 0

print("Normale Transaktionen erzeugt für", N_KUNDEN, "Kund:innen.")

In [ ]:
# --- Verdächtige Muster injizieren (ca. 4 % der Kund:innen) ---
verdaechtige = RNG.choice(range(N_KUNDEN), size=32, replace=False)
extra = []

for i, kid in enumerate(verdaechtige):
    label[kid] = 1
    muster = i % 3

    if muster == 0:   # (A) Structuring: viele Tranchen knapp unter der Schwelle
        n = RNG.integers(8, 15)
        extra.append(pd.DataFrame({
            "kunde_id": kid, "tag": RNG.integers(20, 35, n),
            "betrag": np.round(RNG.uniform(8_500, 9_900, n), 2),
            "land": "CH", "richtung": "ein", "kanal": "Schalter"}))

    elif muster == 1: # (B) Layering: grosser Eingang, schnell weiter ins Hochrisikoland
        gross, t0 = np.round(RNG.uniform(80_000, 250_000), 2), RNG.integers(40, 60)
        extra.append(pd.DataFrame({
            "kunde_id": kid, "tag": [t0, t0+1, t0+2],
            "betrag": [gross, gross*0.49, gross*0.49],
            "land": ["DE", RNG.choice(LAENDER_RISIKO), RNG.choice(LAENDER_RISIKO)],
            "richtung": ["ein", "aus", "aus"], "kanal": "E-Banking"}))

    else:             # (C) Spike: plötzlicher Ausreisser weit über dem eigenen Normal
        n = RNG.integers(3, 6)
        extra.append(pd.DataFrame({
            "kunde_id": kid, "tag": RNG.integers(90, 110, n),
            "betrag": np.round(RNG.uniform(40_000, 120_000, n), 2),
            "land": RNG.choice(LAENDER_RISIKO), "richtung": "aus", "kanal": "E-Banking"}))

tx = pd.concat(zeilen + extra, ignore_index=True)
tx["datum"] = START + pd.to_timedelta(tx["tag"], unit="D")
tx = tx.drop(columns="tag").sort_values(["kunde_id", "datum"]).reset_index(drop=True)
tx["tx_id"] = range(len(tx))
tx["hochrisikoland"] = tx["land"].isin(LAENDER_RISIKO).astype(int)

print(f"Total {len(tx):,} Transaktionen | {tx['kunde_id'].nunique()} Kund:innen "
      f"| {sum(label.values())} davon verdächtig (Ground Truth)")
tx.head(8)

## 3 — Data Understanding (Explorative Datenanalyse, EDA)

Bevor man modelliert, schaut man sich die Daten an: Wie sehen sie aus, wo sind Ausreisser,
was ist plausibel? Diese Phase wird oft unterschätzt — sie frisst in echten Projekten
locker die Hälfte der Zeit.

In [ ]:
# Erster Überblick
print(tx[["betrag", "hochrisikoland"]].describe(), "\n")
print("Transaktionen je Richtung:\n", tx["richtung"].value_counts(), "\n")
print("Transaktionen je Kanal:\n", tx["kanal"].value_counts())

In [ ]:
# Verteilung der Beträge: typische 'long tail'-Form von Finanzdaten
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(tx["betrag"], bins=80)
ax[0].set_title("Beträge (linear) — wenige sehr grosse dominieren")
ax[0].set_xlabel("Betrag (CHF)"); ax[0].set_ylabel("Anzahl")

ax[1].hist(np.log10(tx["betrag"].clip(lower=1)), bins=80)
ax[1].set_title("Beträge (log10) — Struktur wird sichtbar")
ax[1].set_xlabel("log10(Betrag)"); ax[1].set_ylabel("Anzahl")
plt.tight_layout(); plt.show()

**Lesehilfe:** Links erdrückt eine Handvoll Grosstransaktionen das Bild — links unten
klebt alles zusammen. Erst die **Log-Transformation** rechts macht die Struktur sichtbar.
Solche Transformationen sind ein Standard-Trick der Datenaufbereitung.

In [ ]:
# Häufung knapp unter der Meldeschwelle? (Hinweis auf Structuring im Gesamtbild)
nahe = tx[(tx["betrag"] >= 7_000) & (tx["betrag"] <= 11_000)]
plt.figure(figsize=(8, 4))
plt.hist(nahe["betrag"], bins=40)
plt.axvline(MELDESCHWELLE, color="red", linestyle="--", label="Meldeschwelle 10'000")
plt.title("Beträge rund um die Meldeschwelle"); plt.xlabel("Betrag (CHF)")
plt.legend(); plt.tight_layout(); plt.show()

## 4 — Data Preparation (Feature Engineering)

Die Modelle arbeiten nicht auf einzelnen Buchungen, sondern auf einem **Verhaltensprofil
pro Kund:in**. Aus tausenden Transaktionen verdichten wir pro Person eine Handvoll
aussagekräftiger Kennzahlen (*Features*). Gute Features entstehen aus **Fachwissen** —
hier sitzen Compliance-Expert:innen und Data Scientists idealerweise zusammen.

In [ ]:
g = tx.groupby("kunde_id")

feat = pd.DataFrame({
    "tx_anzahl":        g.size(),
    "betrag_summe":     g["betrag"].sum(),
    "betrag_mittel":    g["betrag"].mean(),
    "betrag_max":       g["betrag"].max(),
    "betrag_std":       g["betrag"].std().fillna(0),
    "anteil_ausgehend": g.apply(lambda x: (x["richtung"] == "aus").mean(),
                                include_groups=False),
    "hochrisiko_tx":    g["hochrisikoland"].sum(),
}).reset_index()

feat["label"] = feat["kunde_id"].map(label)   # Ground Truth (in der Praxis selten vorhanden)
print(feat.shape, "Kund:innen x Features")
feat.head()

## 5 — Ansatz A: Regelbasiert

Der klassische Einstieg. Domänenexpert:innen formulieren explizite Wenn-Dann-Regeln.
Vorteil: vollständig nachvollziehbar und prüfbar. Nachteil: starr, leicht umgehbar und
oft sehr viele Fehlalarme.

In [ ]:
# Regel 1: irgendeine Transaktion >= Meldeschwelle
r1 = g["betrag"].max() >= MELDESCHWELLE

# Regel 2: Structuring — >= 5 Transaktionen zwischen 8'500 und 9'999
def ist_structuring(x):
    return ((x["betrag"] >= 8_500) & (x["betrag"] < MELDESCHWELLE)).sum() >= 5
r2 = g.apply(ist_structuring, include_groups=False)

# Regel 3: mindestens eine Transaktion in ein Hochrisikoland
r3 = g["hochrisikoland"].sum() > 0

feat["alert_regel"] = (r1 | r2 | r3).astype(int).values
print("Regelbasierte Alerts:", int(feat["alert_regel"].sum()))

## 6 — Ansatz B: Statistisch (z-Score)

Statt fixer Schwellen messen wir, wie stark eine Transaktion vom **eigenen Normalverhalten**
der Kund:in abweicht. Der **z-Score** sagt: «So viele Standardabweichungen über dem
persönlichen Mittelwert.» Wir nehmen pro Kund:in den maximalen z-Score.

In [ ]:
def max_zscore(x):
    s = x["betrag"]
    if s.std(ddof=0) == 0 or len(s) < 3:
        return 0.0
    return float(((s - s.mean()) / s.std(ddof=0)).max())

feat["max_zscore"] = g.apply(max_zscore, include_groups=False).values

# Naiver Ansatz: einfach z >= 3
feat["alert_z_naiv"] = (feat["max_zscore"] >= 3.0).astype(int)
print("Naive z-Score-Alerts (z >= 3):", int(feat["alert_z_naiv"].sum()),
      "  <- viel zu viele!")

**Warum so viele?** Bei stark schiefen Finanzdaten ist ein «3-Sigma-Maximum» nichts
Besonderes — fast jede:r hat mal einen Ausreisser nach oben. Reine Statistik **ohne
Kontext** flutet uns. Wir kombinieren den z-Score deshalb mit einem fachlichen
**Mindestbetrag**: auffällig *und* materiell relevant.

In [ ]:
BETRAGS_FLOOR = 25_000   # fachliche Untergrenze
feat["alert_statistik"] = ((feat["max_zscore"] >= 3.0) &
                           (feat["betrag_max"] >= BETRAGS_FLOOR)).astype(int)
print("Verfeinerte statistische Alerts:", int(feat["alert_statistik"].sum()))

## 7 — Ansatz C: Unsupervised (Isolation Forest)

Der **Isolation Forest** sucht ohne Vorgabe von Regeln nach **Anomalien**: Datenpunkte,
die sich vom Rest abheben, lassen sich mit wenigen zufälligen Schnitten «isolieren».
Das Modell lernt selbst, was normal ist, und markiert das Ungewöhnliche.
`contamination` ist unsere Schätzung, welcher Anteil anomal ist.

In [ ]:
features = ["tx_anzahl", "betrag_summe", "betrag_mittel", "betrag_max",
            "betrag_std", "anteil_ausgehend", "hochrisiko_tx"]

# Features auf vergleichbare Skala bringen
X = StandardScaler().fit_transform(feat[features])

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X)

feat["anomalie_score"] = -iso.score_samples(X)        # höher = anomaler
feat["alert_iforest"]  = (iso.predict(X) == -1).astype(int)
print("Isolation-Forest-Alerts:", int(feat["alert_iforest"].sum()))

In [ ]:
# Welche Features treiben die auffälligsten Profile? (Top 10 nach Anomalie-Score)
top = feat.sort_values("anomalie_score", ascending=False).head(10)
top[["kunde_id", "tx_anzahl", "betrag_max", "hochrisiko_tx",
     "anomalie_score", "label"]]

## 8 — Evaluation

Jetzt die Stunde der Wahrheit: Wir vergleichen jeden Alert mit der Ground Truth.
Die **Confusion Matrix** zeigt vier Felder — richtig/falsch erkannt:

- **TP** (True Positive): echter Fall, korrekt gemeldet
- **FP** (False Positive): Fehlalarm — kostet Ermittlungszeit
- **FN** (False Negative): übersehener echter Fall — das gefährlichste Feld
- **TN** (True Negative): korrekt als unauffällig durchgelassen

Daraus: **Precision** = TP / (TP+FP) (wie sauber sind die Alerts?),
**Recall** = TP / (TP+FN) (wie viele echte Fälle erwischen wir?),
**F1** = harmonisches Mittel beider.

In [ ]:
def auswerten(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n=== {name} ===")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"  Alerts: {int(y_pred.sum())}  |  Precision={p:.2f}  Recall={r:.2f}  F1={f:.2f}")
    return dict(Ansatz=name, Alerts=int(y_pred.sum()),
                Precision=round(p,2), Recall=round(r,2), F1=round(f,2))

y = feat["label"]
ergebnisse = [
    auswerten(y, feat["alert_regel"],      "Regelbasiert"),
    auswerten(y, feat["alert_statistik"],  "Statistik (z-Score + Floor)"),
    auswerten(y, feat["alert_iforest"],    "Isolation Forest"),
]

In [ ]:
# Confusion Matrix des Isolation Forest als Heatmap
cm = confusion_matrix(y, feat["alert_iforest"])
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1], ["kein Alert", "Alert"]); ax.set_yticks([0,1], ["unauffällig", "verdächtig"])
ax.set_xlabel("Vorhersage"); ax.set_ylabel("Wahrheit"); ax.set_title("Isolation Forest")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=14)
plt.tight_layout(); plt.show()

## 9 — Kombination & Vergleich

Kein Ansatz ist für sich perfekt. In der Praxis kombiniert man sie: z. B. Alert nur, wenn
**mindestens zwei** Ansätze anschlagen. Das hebt die Precision, kann aber Recall kosten.
Welcher Kompromiss richtig ist, entscheidet **nicht** das Modell, sondern die Fachseite —
abhängig von Risikoappetit und Ermittlungskapazität.

In [ ]:
feat["alert_kombi"] = ((feat["alert_regel"] + feat["alert_statistik"]
                        + feat["alert_iforest"]) >= 2).astype(int)
ergebnisse.append(auswerten(y, feat["alert_kombi"], "Kombination (>= 2 Ansätze)"))

vergleich = pd.DataFrame(ergebnisse)
vergleich

In [ ]:
# Alerts und F1 im Vergleich
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(vergleich["Ansatz"], vergleich["Alerts"], color="#4C72B0")
ax[0].set_title("Anzahl Alerts (Arbeitslast für die Analyse)")
ax[0].tick_params(axis="x", rotation=30)

ax[1].bar(vergleich["Ansatz"], vergleich["F1"], color="#55A868")
ax[1].set_title("F1-Score (Balance aus Precision & Recall)")
ax[1].set_ylim(0, 1); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 10 — Aufgaben zum Selber-Ausprobieren

Verändern Sie Parameter, führen Sie die betroffenen Zellen erneut aus und beobachten Sie,
wie sich Alerts und Kennzahlen verschieben. Es gibt **kein** objektiv «richtiges» Ergebnis —
es geht um das Gefühl für die Trade-offs.

**Aufgabe 1 — Sensitivität des Isolation Forest.**
Setzen Sie in Abschnitt 7 `contamination` auf `0.02` bzw. `0.10` und führen Sie
Abschnitt 7 und 8 erneut aus. Wie wandern Precision und Recall? Notieren Sie die Richtung.

**Aufgabe 2 — Schärfe der statistischen Regel.**
Verändern Sie in Abschnitt 6 `BETRAGS_FLOOR` (z. B. 15'000 vs. 50'000). Ab wann kippt der
Ansatz von «zu viele Alerts» zu «zu wenige»?

**Aufgabe 3 — Welcher Fehler ist teurer?**
Diskutieren Sie zu zweit: Was kostet die Bank mehr — ein False Positive oder ein False
Negative? Was bedeutet das für die Wahl zwischen den Ansätzen? (Keine Code-Zelle, Notizen unten.)

**Aufgabe 4 (optional) — Eigenes Feature.**
Bauen Sie in Abschnitt 4 ein zusätzliches Feature, z. B. `tage_aktiv` (Anzahl Tage mit
Aktivität) oder `betrag_range = betrag_max - betrag_min`. Nehmen Sie es in die `features`-
Liste in Abschnitt 7 auf. Verbessert sich der F1-Score?

In [ ]:
# Platz für Aufgabe 4 — eigenes Feature
# Beispiel-Gerüst (auskommentiert):
# feat["tage_aktiv"] = g["datum"].apply(lambda d: d.dt.normalize().nunique()).values
# features.append("tage_aktiv")
# -> danach Abschnitt 7 und 8 erneut ausführen


## Fazit

- **Regeln** sind transparent, aber starr und fehlalarmlastig.
- **Statistik** bringt Kontext (Abweichung vom Eigenverhalten), braucht aber fachliche Leitplanken.
- **Unsupervised ML** findet Unbekanntes und liefert hier den besten Kompromiss — ist
  aber erklärungsbedürftig (Stichwort Modell-Governance, Nachvollziehbarkeit).
- Die **Schwelle** ist eine **Business-Entscheidung**, kein Knopf am Modell.

Der Data Scientist baut den Werkzeugkasten — *was* als Alert gilt, entscheidet die
Fachseite gemeinsam mit ihm. Genau diese Zusammenarbeit zu orchestrieren, ist Ihre Rolle
als Projektleiter:in.